# Lower-back acceleration + gyroscope transport pilot

Tests the lower-back-first research direction using the existing source-aligned Felius/Voisard 18-channel windows. It compares orientation-invariant lower-back acceleration magnitude alone with acceleration-plus-gyroscope magnitude. No new cohort, Sint adapter, or frozen evaluation data are used.

In [1]:
from pathlib import Path
import sys,gc
import numpy as np,pandas as pd,torch
from torch.utils.data import DataLoader,TensorDataset,WeightedRandomSampler
from sklearn.metrics import roc_auc_score,balanced_accuracy_score,brier_score_loss
ROOT=Path.cwd().resolve(); ROOT=ROOT.parent if ROOT.name.lower()=='notebooks' else ROOT
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
from models.stroke_gait_inception import InceptionBlock
P=ROOT/'data'/'processed'; D=torch.device('cuda' if torch.cuda.is_available() else 'cpu'); print('device:',D)
raw=np.load(P/'validated_gait_windows_float32.npy'); m=pd.read_csv(P/'validated_window_metadata.csv'); m=m[m.label.isin(['healthy','stroke'])].reset_index(drop=True); raw=raw[:len(m)]; m['y']=m.label.eq('stroke').astype(int); m['source']=m.dataset_id; m['group']=m.participant_key.astype(str)
x=np.stack([np.linalg.norm(raw[:,:,0:3],axis=2),np.linalg.norm(raw[:,:,3:6],axis=2)],axis=2).astype('float32'); print('input',x.shape,'participants',m.group.nunique())
class Net(torch.nn.Module):
    def __init__(self,c):
        super().__init__(); self.f=torch.nn.Sequential(InceptionBlock(c),torch.nn.MaxPool1d(2),InceptionBlock(64),torch.nn.AdaptiveAvgPool1d(1)); self.c=torch.nn.Sequential(torch.nn.Flatten(),torch.nn.Dropout(.3),torch.nn.Linear(64,1))
    def forward(self,z): return self.c(self.f(z)).squeeze(1)
def w(frame):
    cells=frame.groupby('y').size(); return torch.tensor(frame.y.map(lambda y:1/cells[y]).to_numpy(),dtype=torch.double)
def metric(net,arr,meta,mean,std):
    with torch.inference_mode(): p=torch.sigmoid(net(torch.from_numpy(((arr-mean)/std).transpose(0,2,1).astype('float32')).to(D))).cpu().numpy()
    g=meta.assign(p=p).groupby(['group','y'],as_index=False).p.mean(); return {'participants':len(g),'healthy':int((g.y==0).sum()),'stroke':int((g.y==1).sum()),'auroc':roc_auc_score(g.y,g.p),'balanced_accuracy':balanced_accuracy_score(g.y,g.p>=.5),'healthy_specificity':float((g.loc[g.y==0,'p']<.5).mean()),'brier':brier_score_loss(g.y,g.p)}
rows=[]
for held in sorted(m.source.unique()):
    tr=m.source.ne(held).to_numpy(); va=~tr
    for seed in [42,137,202]:
        for mode,channels in [('lb_acceleration_only',[0]),('lb_acceleration_plus_gyroscope',[0,1])]:
            torch.manual_seed(seed); tx=x[tr][:,:,channels]; mean,std=tx.reshape(-1,len(channels)).mean(0),tx.reshape(-1,len(channels)).std(0).clip(1e-4); z=torch.from_numpy(((tx-mean)/std).transpose(0,2,1).astype('float32')); y=torch.from_numpy(m.loc[tr,'y'].to_numpy('float32')); dl=DataLoader(TensorDataset(z,y),128,sampler=WeightedRandomSampler(w(m.loc[tr]),len(z),replacement=True,generator=torch.Generator().manual_seed(seed+1000)))
            net=Net(len(channels)).to(D); opt=torch.optim.AdamW(net.parameters(),1e-3,weight_decay=1e-4)
            for _ in range(8):
                net.train()
                for a,b in dl: opt.zero_grad(); loss=torch.nn.functional.binary_cross_entropy_with_logits(net(a.to(D)),b.to(D)); loss.backward(); opt.step()
            net.eval(); rows.append({'held_out_source':held,'seed':seed,'mode':mode,**metric(net,x[va][:,:,channels],m.loc[va],mean,std)}); del net,opt,dl,z,y; gc.collect(); torch.cuda.empty_cache() if D.type=='cuda' else None; print('complete',held,seed,mode)
out=pd.DataFrame(rows); out.to_csv(P/'lower_back_accel_gyro_source_transport_pilot.csv',index=False); print(out.groupby(['held_out_source','mode'])[['auroc','balanced_accuracy','healthy_specificity','brier']].agg(['mean','std']).round(4))

device: cuda


input (18511, 500, 2) participants 284


complete felius_2024 42 lb_acceleration_only


complete felius_2024 42 lb_acceleration_plus_gyroscope


complete felius_2024 137 lb_acceleration_only


...

complete voisard_2025 42 lb_acceleration_plus_gyroscope


complete voisard_2025 137 lb_acceleration_only


complete voisard_2025 137 lb_acceleration_plus_gyroscope


complete voisard_2025 202 lb_acceleration_only


complete voisard_2025 202 lb_acceleration_plus_gyroscope
                                                 auroc          \
                                                  mean     std   
held_out_source mode                                             
felius_2024     lb_acceleration_only            0.7825  0.0044   
                lb_acceleration_plus_gyroscope  0.7829  0.0436   
voisard_2025    lb_acceleration_only            0.7357  0.0069   
                lb_acceleration_plus_gyroscope  0.8234  0.0392   

                                               balanced_accuracy          \
                                                            mean     std   
held_out_source mode                                                       
felius_2024     lb_acceleration_only                      0.6978  0.0094   
                lb_acceleration_plus_gyroscope            0.6210  0.0695   
voisard_2025    lb_acceleration_only                      0.6589  0.0241   
                lb_accel

The two-channel lower-back representation advances only if it is non-inferior on both held-out sources and improves at least one transport or calibration metric. A passing pilot justifies a separate Sint 6-DoF adapter audit; it does not replace the current three-channel prototype or touch frozen external data.